# Customer Churn Case Study

This notebook is a compact companion to the production training script. The project goal is to predict telecom churn and convert probabilities into retention decisions.

## Business framing

A false negative means the business misses a customer who is likely to churn. A false positive means the business spends retention budget on a customer who may have stayed anyway. The model is therefore evaluated with recall, ROC AUC, and an ROI-based decision threshold.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.head()

In [ ]:
df.shape, df['Churn'].value_counts(normalize=True).round(3)

## Leakage check

`customerID` is an identifier, not a behavioral signal. It is dropped before modeling so the model learns customer patterns instead of memorizing IDs.

In [ ]:
features = df.drop(columns=['Churn', 'customerID'])
target = df['Churn'].map({'No': 0, 'Yes': 1})
features.columns.tolist()

## Reproducible training

The canonical model training lives in `scripts/train_model.py`. It builds a full sklearn pipeline, compares candidate models, tunes a business threshold, and saves artifacts for the Streamlit app.

In [ ]:
%run ../scripts/train_model.py

In [ ]:
import json

with open('../reports/model_metrics.json', encoding='utf-8') as file:
    metrics = json.load(file)

metrics['selected_model'], metrics['models'][metrics['selected_model']]['roc_auc']